# DistilBERT - Full fine-tuning: fixed hyperparameter search

Честный эксперимент для сравнения с DistilBERT + LoRA. Все конфигурации обучаются на фиксированном train split, лучший checkpoint выбирается только по validation macro-F1, а test split используется единственный раз после выбора модели.

В начало ноутбука добавлено то же Colab/GitHub-подключение, что и в рабочем LoRA-ноутбуке, поэтому оба эксперимента используют одни и те же файлы split из репозитория.


In [1]:
%cd /content
!rm -rf LORA-course-project
!git clone https://github.com/NikitaBaranenkov/LORA-course-project.git
%cd LORA-course-project
!ls
!ls src

/content
Cloning into 'LORA-course-project'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 62 (delta 8), reused 47 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 585.54 KiB | 12.73 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/LORA-course-project
artifacts  data    notebooks  reports		src
configs    legacy  README.md  requirements.txt
constants.py  __init__.py  README.md


In [2]:
%pip uninstall -y torchao
%pip install -q --upgrade peft transformers accelerate datasets evaluate scikit-learn

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.8 MB/s eta 0:00:00


In [3]:
import gc
import inspect
import json
import random
import time
from itertools import product

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from src.constants import (
    BASE_MODEL_DISTILBERT,
    DATASET_NAME,
    FINAL_TEST_EVALUATION_ONLY,
    MAX_LENGTH,
    PROJECT_ROOT,
    REPORTS_TABLES_DIR,
    SEED,
    SELECTION_METRIC,
    TEST_CSV_PATH,
    TRAIN_CSV_PATH,
    VALIDATION_CSV_PATH,
    id2label,
    label2id,
)


In [4]:
MODEL_NAME = BASE_MODEL_DISTILBERT
TRAIN_PATH = TRAIN_CSV_PATH
VAL_PATH = VALIDATION_CSV_PATH
TEST_PATH = TEST_CSV_PATH

RUN_NAME = "distilbert_full_ft_hparam_search"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / RUN_NAME
CHECKPOINTS_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_MODEL_DIR = ARTIFACTS_DIR / "best_model"
RUN_TABLES_DIR = REPORTS_TABLES_DIR / "distilbert_full_ft"

SWEEP_RESULTS_PATH = RUN_TABLES_DIR / "distilbert_full_ft_hparam_search_validation_results.csv"
FINAL_TEST_RESULTS_PATH = RUN_TABLES_DIR / "distilbert_full_ft_best_hparam_test_results.csv"
EXPERIMENT_CONFIG_PATH = RUN_TABLES_DIR / "distilbert_full_ft_hparam_search_config.json"

label_id_to_name = id2label
label_name_to_id = label2id


In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [6]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [7]:
train_df["label"] = train_df["label"].astype(int)
val_df["label"] = val_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

In [8]:
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

raw_datasets = raw_datasets.rename_column("label", "labels")

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
MAX_LENGTH = 128
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )
tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label_name"]
)

tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/8105 [00:00<?, ? examples/s]

Map:   0%|          | 0/1431 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [11]:
raw_dataframes = {
    "train": train_df,
    "validation": val_df,
    "test": test_df
}
sample = tokenized_datasets["train"][0]

print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
label: 1


In [12]:
FULL_FT_GRID = [
    {
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "num_train_epochs": 4,
        "warmup_ratio": 0.06,
    }
    for learning_rate, weight_decay in product([2e-5, 5e-5], [0.0, 0.01])
]

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LOGGING_STEPS = 50


def new_full_ft_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_id_to_name),
        id2label=label_id_to_name,
        label2id=label_name_to_id,
    )


def count_parameters(model):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_share": trainable_params / total_params,
    }


set_seed(SEED)
preview_model = new_full_ft_model()
preview_parameter_summary = count_parameters(preview_model)
print("Number of full fine-tuning configurations:", len(FULL_FT_GRID))
print(f"Total parameters:     {preview_parameter_summary['total_params']:,}")
print(f"Trainable parameters: {preview_parameter_summary['trainable_params']:,}")
print(f"Trainable share:      {100 * preview_parameter_summary['trainable_share']:.2f}%")
del preview_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

pd.DataFrame(FULL_FT_GRID)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of full fine-tuning configurations: 4
Total parameters:     66,955,779
Trainable parameters: 66,955,779
Trainable share:      100.00%


,learning_rate,weight_decay,num_train_epochs,warmup_ratio
0,0.00002,0.00,4,0.06
1,0.00002,0.01,4,0.06
2,0.00005,0.00,4,0.06
3,0.00005,0.01,4,0.06


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


assert SELECTION_METRIC == "macro_f1"


def make_training_arguments(config_id, config):
    training_args_kwargs = {
        "output_dir": str(CHECKPOINTS_DIR / f"config_{config_id:02d}"),
        "learning_rate": config["learning_rate"],
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": config["num_train_epochs"],
        "weight_decay": config["weight_decay"],
        "warmup_ratio": config["warmup_ratio"],
        "logging_steps": LOGGING_STEPS,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": SELECTION_METRIC,
        "greater_is_better": True,
        "save_total_limit": 1,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": torch.cuda.is_available(),
    }
    signature = inspect.signature(TrainingArguments.__init__)
    strategy_key = "eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"
    training_args_kwargs[strategy_key] = "epoch"
    return TrainingArguments(**training_args_kwargs)


In [16]:
# The test split is intentionally absent from hyperparameter selection.
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)

sweep_records = []
best_run = None
for config_id, config in enumerate(FULL_FT_GRID, start=1):
    print(f"\nTraining full fine-tuning configuration {config_id}/{len(FULL_FT_GRID)}:", config)
    set_seed(SEED)
    run_model = new_full_ft_model().to(device)
    parameter_summary = count_parameters(run_model)
    run_trainer = Trainer(
        model=run_model,
        args=make_training_arguments(config_id, config),
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    start_time = time.perf_counter()
    run_trainer.train()
    run_training_time_sec = time.perf_counter() - start_time
    validation_metrics = run_trainer.evaluate(
        tokenized_datasets["validation"],
        metric_key_prefix="validation",
    )
    assert run_trainer.state.best_model_checkpoint is not None

    record = {
        "config_id": config_id,
        **config,
        "val_macro_f1": float(validation_metrics["validation_macro_f1"]),
        "val_weighted_f1": float(validation_metrics["validation_weighted_f1"]),
        "val_accuracy": float(validation_metrics["validation_accuracy"]),
        "training_time_sec": run_training_time_sec,
        "best_checkpoint": run_trainer.state.best_model_checkpoint,
        **parameter_summary,
    }
    sweep_records.append(record)
    if best_run is None or record["val_macro_f1"] > best_run["record"]["val_macro_f1"]:
        best_run = {
            "record": dict(record),
            "config": dict(config),
            "parameter_summary": dict(parameter_summary),
        }

    del run_trainer, run_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sweep_results_df = (
    pd.DataFrame(sweep_records)
    .sort_values("val_macro_f1", ascending=False)
    .reset_index(drop=True)
)
sweep_results_df.to_csv(SWEEP_RESULTS_PATH, index=False)
sweep_results_df



Training full fine-tuning configuration 1/4: {'learning_rate': 2e-05, 'weight_decay': 0.0, 'num_train_epochs': 4, 'warmup_ratio': 0.06}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.417493,0.397172,0.854647,0.804719,0.851432
2,0.323033,0.404591,0.860936,0.810080,0.856216
3,0.177718,0.425974,0.879804,0.843251,0.879841
4,0.080600,0.473721,0.874913,0.835886,0.874255


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.054962,0.593015,4,0.879106,0.838981,0.877866


,config_id,learning_rate,weight_decay,num_train_epochs,warmup_ratio,val_macro_f1,val_weighted_f1,val_accuracy,training_time_sec,best_checkpoint,total_params,trainable_params,trainable_share
0,1,0.00002,0.00,4,0.06,0.843251,0.879841,0.879804,156.822543,/content/LORA-course-project/artifacts/distilb...,66955779,66955779,1.0
1,2,0.00002,0.01,4,0.06,0.842616,0.879728,0.879804,245.148357,/content/LORA-course-project/artifacts/distilb...,66955779,66955779,1.0
2,4,0.00005,0.01,4,0.06,0.838981,0.877866,0.879106,243.035367,/content/LORA-course-project/artifacts/distilb...,66955779,66955779,1.0
3,3,0.00005,0.00,4,0.06,0.838629,0.876420,0.877009,269.644288,/content/LORA-course-project/artifacts/distilb...,66955779,66955779,1.0


In [17]:
assert best_run is not None
best_config = best_run["config"]
selected_training_time_sec = best_run["record"]["training_time_sec"]

print("Best full fine-tuning configuration selected only on validation macro-F1:")
print(best_config)
print("Best checkpoint:", best_run["record"]["best_checkpoint"])
print("Selected configuration training time, sec:", selected_training_time_sec)


Best full fine-tuning configuration selected only on validation macro-F1:
{'learning_rate': 2e-05, 'weight_decay': 0.0, 'num_train_epochs': 4, 'warmup_ratio': 0.06}
Best checkpoint: /content/LORA-course-project/artifacts/distilbert_full_ft_hparam_search/checkpoints/config_01/checkpoint-1521
Selected configuration training time, sec: 156.8225427519999


In [18]:
selected_validation_row = {
    "model": MODEL_NAME,
    "adaptation": "full_fine_tuning",
    **best_config,
    "val_macro_f1": best_run["record"]["val_macro_f1"],
    "val_weighted_f1": best_run["record"]["val_weighted_f1"],
    "val_accuracy": best_run["record"]["val_accuracy"],
    "training_time_sec": selected_training_time_sec,
    "best_checkpoint": best_run["record"]["best_checkpoint"],
}
selected_validation_df = pd.DataFrame([selected_validation_row])
selected_validation_df.to_csv(
    RUN_TABLES_DIR / "distilbert_full_ft_best_hparam_validation_metrics.csv",
    index=False,
)
selected_validation_df


,model,adaptation,learning_rate,weight_decay,num_train_epochs,warmup_ratio,val_macro_f1,val_weighted_f1,val_accuracy,training_time_sec,best_checkpoint
0,distilbert-base-uncased,full_fine_tuning,0.00002,0.0,4,0.06,0.843251,0.879841,0.879804,156.822543,/content/LORA-course-project/artifacts/distilb...


In [19]:
set_seed(SEED)
best_model = AutoModelForSequenceClassification.from_pretrained(
    best_run["record"]["best_checkpoint"]
).to(device)
final_eval_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "final_evaluation"),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
best_trainer = Trainer(model=best_model, args=final_eval_args, compute_metrics=compute_metrics)

BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
best_trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

print("Validation-selected model saved to:", BEST_MODEL_DIR)
print("Loaded checkpoint:", best_run["record"]["best_checkpoint"])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Validation-selected model saved to: /content/LORA-course-project/artifacts/distilbert_full_ft_hparam_search/best_model
Loaded checkpoint: /content/LORA-course-project/artifacts/distilbert_full_ft_hparam_search/checkpoints/config_01/checkpoint-1521


In [20]:
print("DistilBERT full fine-tuning hyperparameter search finished")
print("-" * 60)
print(f"Base model:              {MODEL_NAME}")
print(f"Max length:              {MAX_LENGTH}")
print(f"Configurations trained:  {len(FULL_FT_GRID)}")
print(f"Selected config:         {best_config}")
print(f"Training time, sec:      {selected_training_time_sec:.2f}")
print(f"Best checkpoint:         {best_run['record']['best_checkpoint']}")
print(f"Validation accuracy:     {best_run['record']['val_accuracy']:.5f}")
print(f"Validation macro F1:     {best_run['record']['val_macro_f1']:.5f}")
print(f"Validation weighted F1:  {best_run['record']['val_weighted_f1']:.5f}")


DistilBERT full fine-tuning hyperparameter search finished
------------------------------------------------------------
Base model:              distilbert-base-uncased
Max length:              128
Configurations trained:  4
Selected config:         {'learning_rate': 2e-05, 'weight_decay': 0.0, 'num_train_epochs': 4, 'warmup_ratio': 0.06}
Training time, sec:      156.82
Best checkpoint:         /content/LORA-course-project/artifacts/distilbert_full_ft_hparam_search/checkpoints/config_01/checkpoint-1521
Validation accuracy:     0.87980
Validation macro F1:     0.84325
Validation weighted F1:  0.87984


In [21]:
def softmax_numpy(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def get_main_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted")
    }


def make_per_class_metrics_df(y_true, y_pred):
    target_names = [label_id_to_name[i] for i in range(3)]

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )

    rows = []
    for class_id, class_name in label_id_to_name.items():
        rows.append({
            "class_id": class_id,
            "class_name": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1_score": report[class_name]["f1-score"],
            "support": int(report[class_name]["support"])
        })

    return pd.DataFrame(rows)

In [22]:
validation_main_metrics = {
    "accuracy": best_run["record"]["val_accuracy"],
    "macro_f1": best_run["record"]["val_macro_f1"],
    "weighted_f1": best_run["record"]["val_weighted_f1"],
}
validation_main_metrics


{'accuracy': 0.8798043326345213,
 'macro_f1': 0.8432513007218141,
 'weighted_f1': 0.8798414446195267}

In [23]:
assert FINAL_TEST_EVALUATION_ONLY is True

# This is intentionally the only test-set inference in the notebook.
test_pred_output = best_trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)
test_logits = test_pred_output.predictions
test_probs = softmax_numpy(test_logits)
test_preds = np.argmax(test_probs, axis=1)
test_true = raw_dataframes["test"]["label"].values
test_main_metrics = get_main_metrics(test_true, test_preds)
test_main_metrics


{'accuracy': 0.8806532663316583,
 'macro_f1': 0.8445152695396763,
 'weighted_f1': 0.8815336863152501}

In [24]:
test_classification_report = classification_report(
    test_true,
    test_preds,
    labels=[0, 1, 2],
    target_names=[label_id_to_name[i] for i in range(3)],
    output_dict=True,
    zero_division=0,
)
print("Final test metrics from the validation-selected full fine-tuning checkpoint:")
test_main_metrics


Final test metrics from the validation-selected full fine-tuning checkpoint:


{'accuracy': 0.8806532663316583,
 'macro_f1': 0.8445152695396763,
 'weighted_f1': 0.8815336863152501}

In [25]:
FINAL_RESULT_COLUMNS = [
    "model",
    "adaptation",
    "test_macro_f1",
    "test_weighted_f1",
    "test_accuracy",
    "bearish_f1",
    "bullish_f1",
    "neutral_f1",
    "total_params",
    "trainable_params",
    "trainable_share",
    "training_time_sec",
]

selected_parameter_summary = best_run["parameter_summary"]
final_test_row = {
    "model": MODEL_NAME,
    "adaptation": "full_fine_tuning",
    "test_macro_f1": test_main_metrics["macro_f1"],
    "test_weighted_f1": test_main_metrics["weighted_f1"],
    "test_accuracy": test_main_metrics["accuracy"],
    "bearish_f1": float(test_classification_report["Bearish"]["f1-score"]),
    "bullish_f1": float(test_classification_report["Bullish"]["f1-score"]),
    "neutral_f1": float(test_classification_report["Neutral"]["f1-score"]),
    "total_params": selected_parameter_summary["total_params"],
    "trainable_params": selected_parameter_summary["trainable_params"],
    "trainable_share": selected_parameter_summary["trainable_share"],
    "training_time_sec": selected_training_time_sec,
}
final_test_results_df = pd.DataFrame([final_test_row], columns=FINAL_RESULT_COLUMNS)
assert list(final_test_results_df.columns) == FINAL_RESULT_COLUMNS
final_test_results_df.to_csv(FINAL_TEST_RESULTS_PATH, index=False)
final_test_results_df


,model,adaptation,test_macro_f1,test_weighted_f1,test_accuracy,bearish_f1,bullish_f1,neutral_f1,total_params,trainable_params,trainable_share,training_time_sec
0,distilbert-base-uncased,full_fine_tuning,0.844515,0.881534,0.880653,0.791781,0.82241,0.919355,66955779,66955779,1.0,156.822543


In [26]:
test_per_class_metrics_df = make_per_class_metrics_df(test_true, test_preds)
test_per_class_metrics_df.to_csv(
    RUN_TABLES_DIR / "distilbert_full_ft_best_hparam_test_classification_report.csv",
    index=False,
)
test_per_class_metrics_df


,class_id,class_name,precision,recall,f1_score,support
0,0,Bearish,0.754569,0.832853,0.791781,347
1,1,Bullish,0.825902,0.818947,0.822410,475
2,2,Neutral,0.928944,0.909962,0.919355,1566


In [27]:
class_names = [label_id_to_name[i] for i in range(3)]
test_cm_df = pd.DataFrame(
    confusion_matrix(test_true, test_preds, labels=[0, 1, 2]),
    index=[f"true_{name}" for name in class_names],
    columns=[f"pred_{name}" for name in class_names],
)
test_cm_df.to_csv(
    RUN_TABLES_DIR / "distilbert_full_ft_best_hparam_test_confusion_matrix.csv"
)
test_cm_df


,pred_Bearish,pred_Bullish,pred_Neutral
true_Bearish,289,14,44
true_Bullish,21,389,65
true_Neutral,73,68,1425


In [28]:
test_predictions_df = raw_dataframes["test"].copy()
test_predictions_df["true_label"] = test_true
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(label_id_to_name)
test_predictions_df["pred_label"] = test_preds
test_predictions_df["pred_label_name"] = test_predictions_df["pred_label"].map(label_id_to_name)
test_predictions_df["correct"] = (
    test_predictions_df["true_label"] == test_predictions_df["pred_label"]
)
test_predictions_df["prob_bearish"] = test_probs[:, 0]
test_predictions_df["prob_bullish"] = test_probs[:, 1]
test_predictions_df["prob_neutral"] = test_probs[:, 2]
test_predictions_df.to_csv(
    RUN_TABLES_DIR / "distilbert_full_ft_best_hparam_test_predictions.csv", index=False
)
test_predictions_df.head()


,text,label,label_name,true_label,true_label_name,pred_label,pred_label_name,correct,prob_bearish,prob_bullish,prob_neutral
0,$ALLY - Ally Financial pulls outlook https://t...,0,Bearish,0,Bearish,0,Bearish,True,0.983512,0.007657,0.008831
1,"$DELL $HPE - Dell, HPE targets trimmed on comp...",0,Bearish,0,Bearish,0,Bearish,True,0.980894,0.008637,0.010469
2,$PRTY - Moody's turns negative on Party City h...,0,Bearish,0,Bearish,0,Bearish,True,0.985899,0.006001,0.008099
3,$SAN: Deutsche Bank cuts to Hold,0,Bearish,0,Bearish,0,Bearish,True,0.986347,0.005981,0.007672
4,$SITC: Compass Point cuts to Sell,0,Bearish,0,Bearish,0,Bearish,True,0.987256,0.005558,0.007186


In [29]:
experiment_config = {
    "dataset_name": DATASET_NAME,
    "model": MODEL_NAME,
    "adaptation": "full_fine_tuning",
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "selection_metric": SELECTION_METRIC,
    "selection_split": "validation",
    "test_evaluation_after_validation_selection_only": FINAL_TEST_EVALUATION_ONLY,
    "fixed_grid": FULL_FT_GRID,
    "fixed_training_settings": {
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
    },
    "selected_config": best_config,
    "selected_best_checkpoint": best_run["record"]["best_checkpoint"],
    "selected_training_time_sec": selected_training_time_sec,
    "sweep_results_csv": str(SWEEP_RESULTS_PATH),
    "final_test_results_csv": str(FINAL_TEST_RESULTS_PATH),
}
with EXPERIMENT_CONFIG_PATH.open("w", encoding="utf-8") as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Saved validation sweep table to:", SWEEP_RESULTS_PATH)
print("Saved final best-model test table to:", FINAL_TEST_RESULTS_PATH)
print("Saved validation-selected model to:", BEST_MODEL_DIR)


Saved validation sweep table to: /content/LORA-course-project/reports/tables/distilbert_full_ft/distilbert_full_ft_hparam_search_validation_results.csv
Saved final best-model test table to: /content/LORA-course-project/reports/tables/distilbert_full_ft/distilbert_full_ft_best_hparam_test_results.csv
Saved validation-selected model to: /content/LORA-course-project/artifacts/distilbert_full_ft_hparam_search/best_model


In [30]:
import shutil
from google.colab import files

zip_name = "distilbert_full_ft_hparam_search_results"
shutil.make_archive(zip_name, "zip", root_dir=str(RUN_TABLES_DIR))
files.download(zip_name + ".zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>